# 🤖 Multi-Agent Vietnamese Summarization System
---
**Hệ thống tóm tắt văn bản tiếng Việt đa tác tử**

Notebook này chạy từng bước của workflow và hiển thị output trực quan cho mỗi giai đoạn:
1. 📥 Input & Validation
2. 🔧 Text Processing (underthesea)
3. 📋 Planning Phase (Planner → Observation → Reflection)
4. ⚙️ Execution Phase (Summarize → Verify → Edit)
5. 🏆 Final Output & Verification Report

**🔍 Hệ thống lưu trữ bền vững (Lưu đồng thời vào Notebook & Disk để tối ưu bộ nhớ)**

In [1]:
import sys, os, json, time, importlib
from IPython.display import display, HTML, JSON, Markdown, clear_output
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is on path
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config, llm_loader, vn_tools, session_manager
import agents.base_agent, agents.execution_agent, agents.planner_agent
import agents.execution_observation_agent, agents.execution_reflection_agent

# 🔄 Force-Reload ALL modules to ensure latest bugfixes are applied
for module in [
    config, llm_loader, vn_tools, session_manager, 
    agents.base_agent, agents.planner_agent, agents.execution_agent,
    agents.execution_observation_agent, agents.execution_reflection_agent
]:
    importlib.reload(module)

from session_manager import SessionManager
from llm_loader import LLMManager
from agents.execution_agent import ExecutionAgent # Re-import after reload

# ── Visual Helpers ──
def show_box(title, content, color="#2196F3"):
    display(HTML(f'''
    <div style="border-left:4px solid {color}; padding:12px 16px; margin:8px 0; 
                background:{color}11; border-radius:0 8px 8px 0;">
        <b style="color:{color}; font-size:14px;">{title}</b>
        <div style="margin-top:6px; font-size:13px; line-height:1.6;">{content}</div>
    </div>'''))

def show_json(title, data, color="#9C27B0"):
    html = f'<b style="color:{color};">{title}</b><pre style="background:#1e1e2e; color:#cdd6f4; padding:12px; border-radius:8px; max-height:400px; overflow-y:auto; font-size:12px;">{json.dumps(data, ensure_ascii=False, indent=2)}</pre>'
    display(HTML(html))

def show_status(agent, status, icon="✅"):
    display(HTML(f'<div style="padding:4px 0;font-size:13px;">{icon} <b>{agent}</b>: {status}</div>'))

def show_phase(title, icon="📌"):
    display(HTML(f'<h2 style="color:#e0e0e0; background:linear-gradient(90deg,#1a1a2e,#16213e); padding:12px 20px; border-radius:8px; margin:16px 0;">{icon} {title}</h2>'))

def show_summary_card(summary, word_count, style):
    display(HTML(f'''
    <div style="border:2px solid #4CAF50; border-radius:12px; padding:20px; margin:12px 0; background:#1b2838;">
        <div style="display:flex; gap:16px; margin-bottom:12px;">
            <span style="background:#4CAF50; color:#fff; padding:4px 12px; border-radius:20px; font-size:12px;">📝 {word_count} từ</span>
            <span style="background:#2196F3; color:#fff; padding:4px 12px; border-radius:20px; font-size:12px;">🎨 {style}</span>
        </div>
        <div style="color:#e0e0e0; font-size:14px; line-height:1.8;">{summary}</div>
    </div>'''))

def show_save_confirmation(filename, status="Saved"):
    display(HTML(f'<div style="font-size:11px; color:#4CAF50; text-align:right;">💾 <i>Disk Persistence: {filename} ({status})</i></div>'))

print("✅ All modules & agents reloaded from disk.")

✅ All modules & agents reloaded from disk.


## ⚙️ Cell 2: Configuration

In [2]:
show_phase("Configuration", "⚙️")

RESUME_SESSION = False # Đặt thành True nếu muốn chạy tiếp từ dữ liệu lưu trên Disk

config_info = {
    "Hugging Face Models (Original Qwen Repo)": {
        "Agent (7B-Q4 Shards)": f"{config.AGENT_MODEL_REPO} / {len(config.AGENT_MODEL_FILES)} files",
        "Summarizer (3B-Q8)": f"{config.SUMMARIZER_MODEL_REPO} / {config.SUMMARIZER_MODEL_FILE}",
    },
    "GPU/CPU Optimization": {
        "Resting GPU": "Enabled (2s switch-off)",
        "Persistence (Source of Truth)": "session.json + execution_log.json"
    },
    "Logging Strategy": {
        "Notebook Output": "Enabled (Visual JSON)",
        "Disk Storage": "Enabled (Traceability & Recovery)"
    }
}
show_json("System Configuration", config_info, "#FF9800")

show_status("Model Provider", "Original Qwen Repo (Supporting Split GGUF) 🤗")

## 📥 Cell 3: Load Input Text

In [3]:
show_phase("Load Input Text", "📥")

STYLE = "news_brief"
INPUT_FILE = "sample_input/sample_news.txt"

session = SessionManager()
if not RESUME_SESSION:
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        input_text = f.read().strip()
else:
    if session.load_existing():
        input_text = session.session.get("input_text", "")
        show_status("Session", f"Khôi phục văn bản từ session: {session.session.get('session_id')}")
    else:
        with open(INPUT_FILE, "r", encoding="utf-8") as f:
            input_text = f.read().strip()

word_count = vn_tools.count_words(input_text)
show_box("📄 Input Info", f"<b>File:</b> {INPUT_FILE}<br><b>Số từ:</b> {word_count}<br><b>Style:</b> {STYLE}", "#4CAF50")
show_box("📖 Preview", input_text[:500] + "...", "#607D8B")

## 🔧 Cell 4: Vietnamese Text Processing

In [4]:
show_phase("Vietnamese Text Processing", "🔧")

sentences = vn_tools.split_sentences(input_text)
chunks = vn_tools.chunk_text(sentences)

session.stats["chunks"] = len(chunks)
session._save_session()

show_box("📊 Stats", f"Tổng số câu: <b>{len(sentences)}</b><br>Tổng số chunks: <b>{len(chunks)}</b>", "#2196F3")

for c in chunks:
    show_box(f"Chunk {c['chunk_id']}", f"Sentences: {c['sentence_ids']}<br>Words: {c['word_count']}", "#9C27B0")

## 📋 Cell 5: PHASE 1 – Planning Loop

In [5]:
show_phase("PHASE 1: PLANNING", "📋")

from agents.planner_agent import PlannerAgent

llm = LLMManager()
session = SessionManager()

# 🛠️ Các công cụ cần thiết để Planner có thể xây dựng roadmap đầy đủ
ALL_TOOLS = [
    "sent_tokenize", "chunk_text", "identify_key_points", 
    "summarize_chunk", "merge_summaries", "refine_summary", 
    "verify_claim", "style_edit"
]

if RESUME_SESSION and session.load_existing():
    show_status("Session", f"Khôi phục Session {session.session.get('session_id')} từ Disk...")
    roadmap = session.get_latest_roadmap()
    if roadmap:
        show_status("Planning", "Roadmap đã tồn tại trên Disk. Sử dụng bản mới nhất.")
    else:
        session.increment_stat("planner_loop")
        session.update_phase("planner_agent", session.stats["planner_loop"])
        llm.load_agent_model()
        planner = PlannerAgent()
        ctx = {"input_text": input_text, "style": STYLE, "word_count": word_count, "sentences": sentences, "tools": ALL_TOOLS}
        roadmap = planner.run(ctx)
        session.save_roadmap_version("v1", roadmap)
        session.finalize_roadmap(roadmap)
else:
    session.create_session(input_text, STYLE)
    # Lưu input text vào session để resume
    session.session["input_text"] = input_text
    session._save_session()
    
    session.increment_stat("planner_loop")
    session.update_phase("planner_agent", session.stats["planner_loop"])
    
    llm.load_agent_model()
    planner = PlannerAgent()
    ctx = {"input_text": input_text, "style": STYLE, "word_count": word_count, "sentences": sentences, "tools": ALL_TOOLS}
    show_status("Planner", "Đang xây dựng lộ trình tóm tắt chuyên sâu (Planning)...", "📋")
    roadmap = planner.run(ctx)
    
    # 🏆 Persistence: Lưu đồng thời vào file
    session.save_roadmap_version("v1", roadmap)
    session.finalize_roadmap(roadmap)
    show_save_confirmation("roadmap_versions.json")
    show_save_confirmation("roadmap_final.json")

show_json("Finalized Roadmap (Logic View)", roadmap, "#2196F3")
show_status("Planning", "Lập kế hoạch hoàn tất!", "✅")

[LLMManager] Resolving agent model from HF: Qwen/Qwen2.5-3B-Instruct-GGUF ...


llama_new_context_with_model: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[LLMManager] Agent model loaded.


## ⚙️ Cell 6: PHASE 2 – Execution Loop

In [6]:
show_phase("PHASE 2: EXECUTION", "⚙️")

from agents.execution_agent import ExecutionAgent
from agents.execution_observation_agent import ExecutionObservationAgent
from agents.execution_reflection_agent import ExecutionReflectionAgent

llm = LLMManager()
executor = ExecutionAgent()
exec_obs = ExecutionObservationAgent()
exec_ref = ExecutionReflectionAgent()

tasks = roadmap.get("roadmap", {}).get("tasks", []) or roadmap.get("tasks", [])
state = {"sentences": sentences, "chunks": chunks}

# 💾 Memory Optimization: Khôi phục state từ Disk
if RESUME_SESSION:
    mapping = ["sentences", "chunks", "chunk_summaries", "merged_summary", "refined_summary", "verification_report", "edited_summary", "key_points"]
    for key in mapping:
        val = session.get_data(key)
        if val: state[key] = val
    show_status("Manager", "Dữ liệu workflow đã được khôi phục từ Disk ✅")

for i, task in enumerate(tasks):
    step_id = task.get("step_id", f"step_{i+1}")
    
    # ⏭️ Skip logic (Resume check)
    if RESUME_SESSION and any(e["step_id"] == step_id and e["status"] == "completed" for e in session.execution_log):
        show_status(f"Step {i+1}", f"{task.get('name')} (Skip - Đã hoàn thành)", "⏭️")
        continue
        
    session.increment_stat("execution_loop")
    session.update_phase("execution_agent", session.stats["execution_loop"])
    
    show_status(f"Step {i+1}", f"Thực hiện: <b>{task.get('name')}</b>", "⚙️")
    exec_ctx = {"input_text": input_text, "style": STYLE, "current_step": task, **state}
    output = executor.run(exec_ctx)
    
    # 💾 Cập nhật State & Lưu xuống Disk ngay lập tức
    possible_keys = ["sentences", "chunks", "chunk_summaries", "merged_summary", "refined_summary", "verification_report", "edited_summary", "key_points"]
    for k in possible_keys:
        if k in output and output[k]: 
            state[k] = output[k]
            session.update_data_store(k, output[k])
            
    # 🏆 Ghi Execution Log
    session.log_execution_step(
        step_id=step_id, 
        status="completed" if output.get("status") != "failed" else "failed",
        output_summary=f"Agent: {executor.name} | Action: {output.get('action')}"
    )
    show_save_confirmation("execution_log.json")
    
    # 📺 Hiển thị kết quả bước (Visual)
    show_json(f"Output Bước {i+1}", output, "#4CAF50")
    
    # Nghỉ GPU và chuyển sang model reasoning để đánh giá
    llm.unload()
    llm.load_agent_model()
    
    session.increment_stat("execution_observation_loop")
    session.update_phase("execution_observation_agent", session.stats["execution_observation_loop"])
    o = exec_obs.run({"step_output": output, "input_text": input_text, "style": STYLE})
    
    session.increment_stat("execution_reflection_loop")
    session.update_phase("execution_reflection_agent", session.stats["execution_reflection_loop"])
    r = exec_ref.run({"observation": o, "current_step_idx": i+1, "total_steps": len(tasks)})
    
    show_json("Đánh giá của Reflection Agent", r, "#9C27B0")

[LLMManager] Unloading agent model …
[LLMManager] Resting GPU for 2s (Stability) …
[LLMManager] agent model unloaded.
[LLMManager] Resolving agent model from HF: Qwen/Qwen2.5-3B-Instruct-GGUF ...


llama_new_context_with_model: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[LLMManager] Agent model loaded.


[LLMManager] Unloading agent model …
[LLMManager] Resting GPU for 2s (Stability) …
[LLMManager] agent model unloaded.
[LLMManager] Resolving summarizer model from HF: Qwen/Qwen2.5-3B-Instruct-GGUF ...


llama_new_context_with_model: n_ctx_per_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


...

llama_new_context_with_model: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[LLMManager] Agent model loaded.


[LLMManager] Unloading agent model …
[LLMManager] Resting GPU for 2s (Stability) …
[LLMManager] agent model unloaded.
[LLMManager] Resolving agent model from HF: Qwen/Qwen2.5-3B-Instruct-GGUF ...


llama_new_context_with_model: n_ctx_per_seq (8192) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[LLMManager] Agent model loaded.


## 🏆 Cell 7: Final Result

In [7]:
show_phase("KẾT QUẢ CUỐI CÙNG", "🏆")

session.update_phase("manager", "final")

# Ưu tiên lấy bản tóm tắt hoàn thiện nhất
final_s = state.get("edited_summary") or state.get("refined_summary") or state.get("merged_summary") or "(Tiến trình chưa đạt đến bước tóm tắt)"
show_summary_card(final_s, vn_tools.count_words(final_s), STYLE)

v_report = state.get("verification_report", [])
if not v_report:
    show_status("Cảnh báo", "Chưa có Báo cáo Kiểm chứng. Hãy đảm bảo bước 'Verification' trong roadmap đã chạy thành công.", "⚠️")
else:
    show_json("Báo cáo Kiểm chứng (Verification Report)", v_report, "#FF9800")

if final_s != "(Tiến trình chưa đạt đến bước tóm tắt)":
    # Lưu output cuối cùng vào output.json với đầy đủ các trường yêu cầu
    final_output = {
        "news_id": session.session.get("session_id"),
        "original_article": input_text,
        "summary": final_s,
        "style": STYLE,
        "word_count": vn_tools.count_words(final_s),
        "sentence_count": len(sentences),
        "chunks": session.stats.get("chunks", 0),
        "planner_stats": {
            "planner_loop": session.stats.get("planner_loop", 0),
            "planner_observation_loop": session.stats.get("planner_observation_loop", 0),
            "planner_reflection_loop": session.stats.get("planner_reflection_loop", 0)
        },
        "execution_stats": {
            "execution_loop": session.stats.get("execution_loop", 0),
            "execution_observation_loop": session.stats.get("execution_observation_loop", 0),
            "execution_reflection_loop": session.stats.get("execution_reflection_loop", 0)
        },
        "verification_report": v_report
    }
    session.save_output(final_output)
    show_save_confirmation("output.json")
    show_status("Hệ thống", "Dữ liệu đã được lưu trữ bền vững tại thư mục /data.", "✅")

## 📊 Cell 8: Diagnostic Persistence View

In [8]:
show_phase("Persistence Diagnostic", "📊")

if session.load_existing():
    exec_log = session.execution_log
    roadmap_log = session.roadmap_versions
    stats = session.stats

    show_box("🔍 Nội dung execution_log.json", f"Số bước đã lưu: <b>{len(exec_log)}</b>", "#4CAF50")
    display(JSON(exec_log))

    show_box("📊 Toàn bộ thông số Thống kê (Stats)", f"Lưu trong session.json", "#FF9800")
    display(JSON(stats))
    
    show_box("🔍 Nội dung roadmap_versions.json", f"Số phiên bản kế hoạch: <b>{len(roadmap_log)}</b>", "#2196F3")
    display(JSON(roadmap_log))
else:
    show_status("Lỗi", "Không tìm thấy file session.json để chẩn đoán.", "❌")

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>